In [5]:
import os
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy import create_engine

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector

dotenv_path = Path('/Users/sir/Desktop/Project/RAG/.env')

# Load .env file
load_dotenv(dotenv_path=dotenv_path)

# Read PostgreSQL credentials from environment
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST", "localhost")   # default if not set
DBNAME = os.getenv("DB_NAME")
PORT = os.getenv("DB_PORT", "5432")      # default if not set

# Construct the database URL
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?options=-csearch_path%3Dvector,public"
# SQLAlchemy engine
engine = create_engine(DATABASE_URL)

In [7]:
import os
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy import create_engine, text

# Load langchain modules
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector

# --- Your Setup Code ---
dotenv_path = Path('/Users/sir/Desktop/Project/RAG/.env')
load_dotenv(dotenv_path=dotenv_path)

# Read PostgreSQL credentials from environment
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST", "localhost")
DBNAME = os.getenv("DB_NAME")
PORT = os.getenv("DB_PORT", "5432")

# Check if essential variables are loaded
if not all([USER, PASSWORD, DBNAME]):
    print("❌ ERROR: DB_USER, DB_PASSWORD, or DB_NAME not loaded from .env file.")
    exit()

# Construct the database URL
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?options=-csearch_path%3Dvector,public"

# SQLAlchemy engine
engine = create_engine(DATABASE_URL)

In [8]:
## 🚀 Test the Database Connection

try:
    with engine.connect() as connection:
        # Execute a simple query to confirm connectivity
        result = connection.execute(text("SELECT 1"))
        
        # Check if the result is 1
        if result.scalar() == 1:
            print("✅ **Connection Successful!**")
            print(f"Database: {DBNAME} on {HOST}:{PORT}")
            print(f"User: {USER}")
        else:
            print("⚠️ Connection established, but test query failed.")

except Exception as e:
    print("❌ **Connection Failed!**")
    print(f"Error details: {e}")
    print("\nTroubleshooting Tips:")
    print("* Double-check the values in your `.env` file.")
    print("* Ensure PostgreSQL is running and accepting connections.")
    print(f"* Verify the user `{USER}` has the correct password and login permission.")

✅ **Connection Successful!**
Database: LLM on localhost:5432
User: rag_client


In [ ]:
# Load local embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="/Users/sir/Downloads/HuggingFace/sentence_transformer/all-mpnet-base-v2"
)

# Initialize PGVector — metadata stored automatically in JSONB column
vector_store = PGVector(
    embeddings=embeddings,       # embedding function
    collection_name="collection", # table name (will become langchain_pg_collection)
    connection=engine,
    # pre_delete_collection=True,  # If True, will delete the collection if it exists.
    use_jsonb=True,              # enables storing metadata per vector
)

Collection not found


In [16]:
# Add texts with metadata
vector_store.add_texts(
    [
        "AI improves workforce analytics",
        "Predictive models reduce employee attrition",
        "Sentiment analysis of employee surveys"
    ],
    metadatas=[
        {"source": "HR Report 2025", "topic": "analytics"},
        {"source": "HR Report 2025", "topic": "attrition"},
        {"source": "Survey Data", "topic": "sentiment"}
    ]
)



['8b6d6bbf-0edb-4861-b938-c6a46c032a4c',
 '69f7f218-ac7b-45fe-9503-490df69af5af',
 'd83b2512-f1b3-49cf-86a3-2aec670ae85e']

In [15]:
# Semantic search
results = vector_store.similarity_search("employee engagement and AI", k=3)
for r in results:
    print(r.page_content, r.metadata)

AI improves workforce analytics {'topic': 'analytics', 'source': 'HR Report 2025'}
Predictive models reduce employee attrition {'topic': 'attrition', 'source': 'HR Report 2025'}
Sentiment analysis of employee surveys {'topic': 'sentiment', 'source': 'Survey Data'}
